# StockFries Data Analysis
This notebook loads the raw 13F Hedge Fund data from StockFries, filters for the most recent reporting period, and visualizes the "heaviest" aggregated positions (most owned across tracked hedge funds). This replicates the "Heaviest" logic used on the StockFries homepage.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")

# Load the dataset
# If running on Kaggle, this path might need to be adjusted to:
# df = pd.read_csv('/kaggle/input/hedge-fund-13f-positions-from-stockfries-2018-2026/data_by_stocks.csv')
# For local usage, we load directly from the zip/csv file.
try:
    # Try local zip first (pandas 1.2+ supports reading zip natively)
    df = pd.read_csv('data_by_stocks.zip')
except FileNotFoundError:
    try:
        # Fallback to local csv
        df = pd.read_csv('data_by_stocks.csv')
    except FileNotFoundError:
        print("Data file not found. Ensure the dataset is in the working directory.")

# Display the first few rows
df.head()

In [ ]:
# Ensure 'Report Period' is datetime format for accurate sorting/filtering
df['Report Period'] = pd.to_datetime(df['Report Period'])

# Find the most recent reporting period
most_recent_period = df['Report Period'].max()
print(f"Most recent reporting period: {most_recent_period.strftime('%Y-%m-%d')}")

# Filter data for only the most recent period
df_recent = df[df['Report Period'] == most_recent_period]
print(f"Total positions held in the most recent period: {len(df_recent)}")

In [ ]:
# Aggregate the data to find the "Heaviest" positions (Sum of Values per Stock)
# Note: 'Value (x$1000)' is the raw SEC filing unit. We sum this across all funds.
heaviest_agg = df_recent.groupby('Name of Issuer')['Value (x$1000)'].sum()

# Convert to Millions ($ MMs) for readability by dividing by 1000, then sort descending
heaviest_agg = (heaviest_agg / 1000).sort_values(ascending=False).round(1)

# Get the Top 20 Heaviest Stocks
top_20_heaviest = heaviest_agg.head(20)

# Display as a dataframe
top_20_df = top_20_heaviest.reset_index()
top_20_df.columns = ['Stock / Issuer', 'Value ($ MMs)']
display(top_20_df)

In [ ]:
# Visualize the Top 20 Heaviest Positions
plt.figure(figsize=(12, 8))

# Create a horizontal bar plot
barplot = sns.barplot(
    x='Value ($ MMs)', 
    y='Stock / Issuer', 
    data=top_20_df, 
    palette='viridis'
)

# Add labels and title
plt.title(f'Top 20 Heaviest Hedge Fund Holdings ({most_recent_period.strftime("%b %Y")})', fontsize=16, pad=15)
plt.xlabel('Aggregated Value ($ MMs)', fontsize=12)
plt.ylabel('Stock / Issuer', fontsize=12)

# Add value annotations to the end of each bar
for i, p in enumerate(barplot.patches):
    width = p.get_width()
    plt.text(width + (width * 0.02), p.get_y() + p.get_height() / 2, f'${width:,.1f}M', 
             ha='left', va='center', fontsize=10)

# Adjust layout to prevent cutoff
plt.tight_layout()
plt.show()